In [1]:
%load_ext autoreload
%autoreload 2
import torch
import math
torch.manual_seed(19)
from src.retrievers.models.encoder import attention, MultiHeadAttention, FeedForward, EncoderLayer

In [2]:
##### TEST IF WORKS,  DELETE
Q = torch.randn(3,4)
K = torch.randn(3,4)
V = torch.randn(3,4)

# Run the attention function
X_prime, attention_weights = attention(Q, K, V)

# Check 1: form, skal være torch.Size([3, 4]): 3 ord, 4 tal pr. ord
print("Form af X_prime:", X_prime.shape)

# Check 2: hver række i attention skal summe til 1 (Notebook 12.1: "should be positive sum to one")
print("Rækkesummer:", attention_weights.sum(dim=-1))

# Selve resultatet
print(X_prime)

Form af X_prime: torch.Size([3, 4])
Rækkesummer: tensor([1., 1., 1.])
tensor([[-0.3819,  0.3114, -0.3864, -1.3563],
        [ 0.3922, -0.5987, -0.4263, -1.1151],
        [-0.1947, -0.0974, -0.4804, -1.2937]])


In [3]:
Q = torch.randn(2,2,3,4)
K = torch.randn(2,2,3,4)
V = torch.randn(2,2,3,4)

In [4]:
X_prime, attention_weights = attention(Q,K,V)

print("Form af X_prime:", X_prime.shape)               # skal være (2, 2, 3, 4)
print("Rækkesummer:", attention_weights.sum(dim=-1))    # skal være lutter 1-taller

Form af X_prime: torch.Size([2, 2, 3, 4])
Rækkesummer: tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]])


In [5]:
# Test 2
# Nu prøver vi en mask
mask = torch.tensor([[1,1,1],[1,1,0]])
# Her har vi en form på (2, 3), 2 tekster og 3 pladser.
# Men vi ved at attention har formen (2,2,3,3). Så vi bruger unsqueeze til at udligne den her forskel. Ligesom i week 6 notebook.

mask = mask.unsqueeze(1).unsqueeze(1)
X_prime, attention_weights = attention(Q, K, V, mask=mask)
print("Tekst 2, hoved 1:\n", attention_weights[1, 0])

Tekst 2, hoved 1:
 tensor([[0.6553, 0.3447, 0.0000],
        [0.6440, 0.3560, 0.0000],
        [0.6640, 0.3360, 0.0000]])


In [6]:
# test 3
x = torch.randn(2, 3, 8)              # 2 texts, 3 words, 8 numbers per word
mask = torch.tensor([[1, 1, 1],
                     [1, 1, 0]])      # text 2 has padding on the last spot

mha = MultiHeadAttention(8, 2)
out = mha(x, mask)

print("out:", out.shape)                                   # skal være (2, 3, 8): samme form som x
print("weights:", mha.attention_weights.shape)             # skal være (2, 2, 3, 3)
print("tekst 2, hoved 1:\n", mha.attention_weights[1, 0])  # sidste kolonne skal være 0

out: torch.Size([2, 3, 8])
weights: torch.Size([2, 2, 3, 3])
tekst 2, hoved 1:
 tensor([[0.5469, 0.4531, 0.0000],
        [0.3606, 0.6394, 0.0000],
        [0.4511, 0.5489, 0.0000]], grad_fn=<SelectBackward0>)


In [7]:
# Test 4
ffn = FeedForward(8, 32)
x = torch.randn(2, 3, 8)
print(ffn(x).shape)     # torch.Size([2, 3, 8])
print(ffn)

torch.Size([2, 3, 8])
FeedForward(
  (linear1): Linear(in_features=8, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=8, bias=True)
)


In [8]:
# Test 5
layer = EncoderLayer(8, 2, 32)
x = torch.randn(2, 3, 8)
mask = torch.tensor([[1, 1, 1],
                     [1, 1, 0]])
print(layer(x, mask).shape)     # skal være torch.Size([2, 3, 8])
print(layer)                    # viser alle byggeklodserne

torch.Size([2, 3, 8])
EncoderLayer(
  (attention): MultiHeadAttention(
    (linear1): Linear(in_features=8, out_features=8, bias=True)
    (linear2): Linear(in_features=8, out_features=8, bias=True)
    (linear3): Linear(in_features=8, out_features=8, bias=True)
    (linear4): Linear(in_features=8, out_features=8, bias=True)
  )
  (feed_forward): FeedForward(
    (linear1): Linear(in_features=8, out_features=32, bias=True)
    (linear2): Linear(in_features=32, out_features=8, bias=True)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
)


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

batch = tokenizer(["hund bider mand", "hej verden"], padding=True, return_tensors="pt")
print(batch["input_ids"])         # token-id'er, (2, N)
print(batch["attention_mask"])    # 1 = ord, 0 = padding
print(tokenizer.convert_ids_to_tokens(batch["input_ids"][0]))   # se tokens som tekst
print(tokenizer.vocab_size)       # 30522

tensor([[  101, 15876,  4859,  7226,  2121,  2158,  2094,   102],
        [  101,  2002,  3501, 16184,  2078,   102,     0,     0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0]])
['[CLS]', 'hu', '##nd', 'bid', '##er', 'man', '##d', '[SEP]']
30522


In [ ]:
import torch.nn as nn

class Embeddings(torch.nn.Module):
    def __init__(self,vocab_size, emb_dim, max_len):
        super(Embeddings, self).__init__()

        # Vi opretter en matrix som fungere som ordbog med en række pr. token i hele ordforrådet.
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)

        # Vi opretter en lignende tabel men med en række for hver plads i teksten.
        self.position_embedding = nn.Embedding(max_len, emb_dim)

    def forward(self, token_ids):
        # Token_ids er (B, N)

        # B = hvor mange tekster, N = antal tokens
        B, N = token_ids.shape 

        # Tokens får positions nummer fra 0 indeksering og op
        positions = torch.arange(N)

        # Slår først op i "ordbogen" hvad der står, så slår vi op hvor står det henne. Det lægger man sammen 
        # så hver vektor har en værdi som både er hvad og hvor det står.
        x = self.token_embedding(token_ids) + self.position_embedding(positions)

        return x


In [14]:
# TEST 6: Embeddings
emb = Embeddings(vocab_size=100, emb_dim=8, max_len=16)
token_ids = torch.randint(0, 100, (2, 3))      # 2 texts, 3 tokens
x = emb(token_ids)
print(token_ids.shape, "->", x.shape)          # (2, 3) -> (2, 3, 8)

torch.Size([2, 3]) -> torch.Size([2, 3, 8])


In [15]:
emb = Embeddings(vocab_size=tokenizer.vocab_size, emb_dim=8, max_len=256)
x = emb(batch["input_ids"])
print(batch["input_ids"].shape, "->", x.shape)

torch.Size([2, 8]) -> torch.Size([2, 8, 8])


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, max_len, n_layers, n_heads, emb_dim, feed_forward_dim, pooling_type="cls"):
        super(TransformerEncoder, self).__init__()

        # 
        self.pooling_type = pooling_type

        #
        self.embeddings = Embeddings(vocab_size, emb_dim, max_len)

        self.layers = nn.ModuleList([EncoderLayer(emb_dim, n_heads, feed_forward_dim) for _ in range(n_layers)])

    def forward(self, token_ids, attention_mask=None):

        x = self.embeddings(token_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return x